# 05 — Joins & Unions

**What you will learn:**
- All join types: inner, left, right, full outer, left semi, left anti, cross
- Joining on single and multiple keys
- Handling column name conflicts after a join
- Broadcast join hint for small tables
- `union()`, `unionByName()` — stacking DataFrames
- Self join

## Setup — Three Related DataFrames

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, broadcast

# Employees
emp_data = [
    (1, "Alice",   10, 95000.0),
    (2, "Bob",     20, 72000.0),
    (3, "Charlie", 10, 105000.0),
    (4, "Diana",   30, 68000.0),
    (5, "Eve",     20, 78000.0),
    (6, "Frank",   99, 88000.0),   # dept 99 does not exist
]
df_emp = spark.createDataFrame(emp_data, ["emp_id", "name", "dept_id", "salary"])

# Departments
dept_data = [
    (10, "Engineering", "NYC"),
    (20, "Marketing",   "SF"),
    (30, "HR",          "NYC"),
    (40, "Finance",     "Chicago"),  # no employees in Finance
]
df_dept = spark.createDataFrame(dept_data, ["dept_id", "dept_name", "city"])

# Orders
order_data = [
    (101, 1, 500.0),
    (102, 2, 200.0),
    (103, 1, 750.0),
    (104, 5, 300.0),
    (105, 7, 450.0),  # emp_id 7 does not exist in employees
]
df_orders = spark.createDataFrame(order_data, ["order_id", "emp_id", "amount"])

print("Employees:"); df_emp.show()
print("Departments:"); df_dept.show()
print("Orders:"); df_orders.show()

## 1. INNER JOIN — Only Matching Rows

Returns rows where the join key exists in BOTH DataFrames.

In [0]:
df_inner = df_emp.join(df_dept, on="dept_id", how="inner")
df_inner.show()
print("Inner join count:", df_inner.count())  # Frank (dept 99) excluded
                                              # Finance (dept 40) excluded

## 2. LEFT JOIN — All Left Rows + Matching Right

All rows from the left DataFrame; nulls for non-matching right rows.

In [0]:
df_left = df_emp.join(df_dept, on="dept_id", how="left")
df_left.show()
# Frank appears with null dept_name / city because dept 99 doesn't exist

## 3. RIGHT JOIN — Matching Left + All Right Rows

In [0]:
df_right = df_emp.join(df_dept, on="dept_id", how="right")
df_right.show()
# Finance appears with null employee columns because no one belongs to it

## 4. FULL OUTER JOIN — All Rows from Both Sides

In [0]:
df_full = df_emp.join(df_dept, on="dept_id", how="full")
df_full.show()
# Frank (dept 99) shows with nulls on right
# Finance (dept 40) shows with nulls on left

## 5. LEFT SEMI JOIN — Left Rows That Have a Match (No Right Columns)

Equivalent to: `WHERE dept_id IN (SELECT dept_id FROM dept)`

In [0]:
df_semi = df_emp.join(df_dept, on="dept_id", how="left_semi")
df_semi.show()
# Returns only employee columns; Frank excluded (dept 99 not in dept table)

## 6. LEFT ANTI JOIN — Left Rows That Have NO Match

Equivalent to: `WHERE dept_id NOT IN (SELECT dept_id FROM dept)`

In [0]:
df_anti = df_emp.join(df_dept, on="dept_id", how="left_anti")
df_anti.show()
# Returns only Frank — the one employee whose dept_id has no match

## 7. CROSS JOIN — Every Row × Every Row (Cartesian Product)

No join key. Returns M × N rows. Use with extreme care.

In [0]:
# Cross join a small lookup table
labels = spark.createDataFrame([("Q1",), ("Q2",), ("Q3",), ("Q4",)], ["quarter"])

df_cross = df_emp.crossJoin(labels)
print("Cross join count:", df_cross.count())   # 6 employees × 4 quarters = 24
df_cross.show()

## 8. Joining on Multiple Keys

In [0]:
# Create a composite-key example
sales_data = [
    (1, 2023, 50000.0),
    (2, 2023, 30000.0),
    (1, 2022, 45000.0),
    (3, 2023, 70000.0),
]
df_sales = spark.createDataFrame(sales_data, ["emp_id", "year", "sales_amount"])

target_data = [
    (1, 2023, 48000.0),
    (2, 2023, 32000.0),
    (1, 2022, 47000.0),
]
df_target = spark.createDataFrame(target_data, ["emp_id", "year", "target_amount"])

# Join on both emp_id AND year
df_multi = df_sales.join(df_target, on=["emp_id", "year"], how="left")
df_multi.show()

## 9. Handling Duplicate Column Names After Join

When both DataFrames have a column with the same name (other than the join key),
the result has ambiguous columns. Fix with aliases or drop.

In [0]:
# Both df_emp and df_orders have "emp_id"
# Use explicit column expressions to avoid ambiguity
df_emp_alias  = df_emp.alias("e")
df_orders_alias = df_orders.alias("o")

df_joined = df_emp_alias.join(
    df_orders_alias,
    col("e.emp_id") == col("o.emp_id"),
    how="left"
)

# Select with alias prefix to disambiguate
df_joined.select(
    col("e.emp_id").alias("emp_id"),
    col("e.name"),
    col("o.order_id"),
    col("o.amount")
).show()

In [0]:
# Alternative: drop the duplicate column after join
df_joined2 = df_emp.join(df_orders, df_emp["emp_id"] == df_orders["emp_id"], how="left")
df_clean = df_joined2.drop(df_orders["emp_id"])   # drop right side's emp_id
df_clean.show()

## 10. Broadcast Join (Hint for Small Tables)

When one table is small, tell Spark to broadcast it to every executor.
This avoids a shuffle (sort-merge join) and is much faster.

In [0]:
# Broadcast the small departments table
df_broadcast = df_emp.join(broadcast(df_dept), on="dept_id", how="inner")
df_broadcast.show()

# Spark also auto-broadcasts if the table is below spark.sql.autoBroadcastJoinThreshold
# (default 10 MB). You can also configure via SQL hint: /*+ BROADCAST(t) */

## 11. Self Join — Join a DataFrame with Itself

In [0]:
# Find each employee's manager name
emp_manager_data = [
    (1, "Alice",   10),
    (2, "Bob",     10),
    (3, "Charlie", None),  # no manager (top-level)
    (10, "David",  None),  # manager
]
df_self = spark.createDataFrame(emp_manager_data, ["emp_id", "name", "manager_id"])

mgr = df_self.alias("mgr")
emp = df_self.alias("emp")

df_with_manager = (
    emp.join(mgr, col("emp.manager_id") == col("mgr.emp_id"), how="left")
    .select(
        col("emp.emp_id"),
        col("emp.name").alias("employee"),
        col("mgr.name").alias("manager")
    )
)
df_with_manager.show()

## 12. union() and unionByName()

In [0]:
# union() — stacks rows; columns matched by POSITION (not name)
dept_a = spark.createDataFrame([(1, "Alice", "Eng")], ["id", "name", "dept"])
dept_b = spark.createDataFrame([(2, "Bob",   "Mkt")], ["id", "name", "dept"])

df_union = dept_a.union(dept_b)
df_union.show()

In [0]:
# unionByName() — stacks rows by COLUMN NAME (safe when order differs)
df_x = spark.createDataFrame([(1, "Alice")], ["id", "name"])
df_y = spark.createDataFrame([("Bob", 2)],   ["name", "id"])   # columns swapped

# union() would give wrong results here — would put "Bob" in id column
df_safe = df_x.unionByName(df_y)
df_safe.show()

In [0]:
# unionByName with allowMissingColumns=True — missing columns filled with null
df_a = spark.createDataFrame([(1, "Alice", "Eng")], ["id", "name", "dept"])
df_b = spark.createDataFrame([(2, "Bob")],           ["id", "name"])

df_merged = df_a.unionByName(df_b, allowMissingColumns=True)
df_merged.show()

## Join Type Summary

| Join Type | how= | Returns |
|---|---|---|
| Inner | `"inner"` | Rows matching in both |
| Left Outer | `"left"` | All left + matched right (nulls for no match) |
| Right Outer | `"right"` | Matched left + all right (nulls for no match) |
| Full Outer | `"full"` | All rows from both sides |
| Left Semi | `"left_semi"` | Left rows that have a match (no right columns) |
| Left Anti | `"left_anti"` | Left rows that have NO match |
| Cross | `crossJoin()` | Every left row × every right row |

**Performance tips:**
- Use `broadcast()` for tables under ~10 MB
- Avoid cross joins on large tables
- Always alias DataFrames when joining on the same column name